
# Cyclistic Case Study – Process

This notebook documents the **data cleaning, transformation, and validation steps** performed to prepare the Cyclistic trip data for analysis.

## Process Overview

The goal of the processing phase was to transform raw monthly trip data into an analysis‑ready dataset.  
All steps were performed using reproducible Python code, and the final dataset was validated before being loaded into Microsoft SQL Server.

## Data Ingestion and Consolidation

Monthly CSV files were loaded directly from compressed ZIP archives and merged into a single dataset.  
All files followed a consistent schema, allowing them to be combined without manual intervention.

Ride identifiers were used to detect and remove duplicate records to ensure data integrity.

In [13]:

# This script reads all the zip files in the data directory, extracts the CSV files, and concatenates them into a single DataFrame.
import pandas as pd
import glob
import zipfile

paths = glob.glob("data/*/*.zip")
# create a list of dataframes, one for each zip file
dfs = []

for f in paths:
    with zipfile.ZipFile(f, 'r') as z:
        csv_files = [
            name for name in z.namelist()
            if name.endswith('.csv') and not name.startswith('__MACOSX')
        ]

        if len(csv_files) == 0:
            raise ValueError(f"No CSV found in {f}")
        if len(csv_files) > 1:
            raise ValueError(f"Multiple CSVs found in {f}: {csv_files}")

        with z.open(csv_files[0]) as csv_file:
            df = pd.read_csv(csv_file)
    # ===========================

  
    dfs.append(df)
# concatenate all dataframes into one
df_all = pd.concat(dfs, ignore_index=True)
print(df_all.shape)

(32167289, 13)


In [14]:
df_all.tail()


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
32167284,1ADE06C58744ECAB,electric_bike,2026-03-02 09:11:49.421,2026-03-02 09:16:37.134,Western Ave & Lunt Ave,CHI00801,Clark St & Lunt Ave,CHI00642,42.008594,-87.690492,42.009011,-87.674112,member
32167285,87819D83AF437454,classic_bike,2026-03-05 07:38:53.806,2026-03-05 07:44:13.659,Western Ave & Lunt Ave,CHI00801,Clark St & Lunt Ave,CHI00642,42.008594,-87.690492,42.009011,-87.674112,member
32167286,7E910EFCCBCF8440,electric_bike,2026-03-25 18:27:59.727,2026-03-25 18:35:06.921,DuSable Lake Shore Dr & Belmont Ave,CHI00480,Southport Ave & Waveland Ave,CHI00336,41.940775,-87.639192,41.948226,-87.664071,casual
32167287,577CC5F21A42A372,electric_bike,2026-03-09 16:26:54.043,2026-03-09 16:29:55.413,Western Ave & Lunt Ave,CHI00801,Clark St & Lunt Ave,CHI00642,42.008594,-87.690492,42.009011,-87.674112,member
32167288,EAD156B889B86C8E,electric_bike,2026-03-24 08:20:16.530,2026-03-24 08:29:20.189,Racine Ave & Webster Ave,CHI02144,Southport Ave & Waveland Ave,CHI00336,41.921600,-87.658590,41.948226,-87.664071,member


In [17]:
df_all.describe()

,start_lat,start_lng,end_lat,end_lng
count,3.216729e+07,3.216729e+07,3.213204e+07,3.213204e+07
mean,4.190259e+01,-8.764656e+01,4.190291e+01,-8.764677e+01
std,4.517164e-02,2.793482e-02,5.345465e-02,7.502019e-02
min,4.163000e+01,-8.794000e+01,0.000000e+00,-1.440500e+02
25%,4.188114e+01,-8.766000e+01,4.188132e+01,-8.766000e+01
50%,4.189897e+01,-8.764301e+01,4.189964e+01,-8.764312e+01
75%,4.193000e+01,-8.762952e+01,4.193000e+01,-8.762954e+01
max,4.563503e+01,-7.379648e+01,8.796000e+01,1.525300e+02


In [18]:
df_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 32167289 entries, 0 to 32167288
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             str    
 1   rideable_type       str    
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  str    
 5   start_station_id    object 
 6   end_station_name    str    
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       str    
dtypes: float64(4), object(2), str(7)
memory usage: 3.1+ GB


In [20]:
df_all.isnull().sum()

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    4867265
start_station_id      4868020
end_station_name      5145167
end_station_id        5145769
start_lat                   0
start_lng                   0
end_lat                 35248
end_lng                 35248
member_casual               0
dtype: int64

In [ ]:
df_all.drop_duplicates(inplace=True)

# no duplicates, so no change in shape

In [ ]:
df_all.shape


(32167289, 13)

## Timestamp Standardization and Trip Duration

Trip start and end time columns were converted to datetime format to enable accurate time‑based calculations.

Trip duration was calculated in minutes based on the difference between start and end times.


### Invalid records

Trips with negative or zero duration were identified during data validation.  
These records were removed, as they indicate invalid timestamps and do not represent real user behavior.

In [ ]:
# time standardization
import datetime as dt

# remove milliseconds from started_at and ended_at
df_all['started_at'] = df_all['started_at'].str.split('.', n=1).str[0]
df_all['ended_at'] = df_all['ended_at'].str.split('.', n=1).str[0]

# convert started_at and ended_at to datetime
df_all['started_at'] = pd.to_datetime(df_all['started_at'])
df_all['ended_at'] = pd.to_datetime(df_all['ended_at'])



In [ ]:
# calculate duration in minutes
df_all['duration'] = (df_all['ended_at'] - df_all['started_at']).dt.total_seconds() / 60

In [33]:
df_all['duration [min]'] = df_all['duration'].astype('int')
df_all.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration,duration [min]
0,A847FADBBC638E45,docked_bike,2020-04-26 17:45:14,2020-04-26 18:12:03,Eckhart Park,86,Lincoln Ave & Diversey Pkwy,152.0,41.8964,-87.6610,41.9322,-87.6586,member,26,26
1,5405B80E996FF60D,docked_bike,2020-04-17 17:08:54,2020-04-17 17:17:03,Drake Ave & Fullerton Ave,503,Kosciuszko Park,499.0,41.9244,-87.7154,41.9306,-87.7238,member,8,8
2,5DD24A79A4E006F4,docked_bike,2020-04-01 17:54:13,2020-04-01 18:08:36,McClurg Ct & Erie St,142,Indiana Ave & Roosevelt Rd,255.0,41.8945,-87.6179,41.8679,-87.6230,member,14,14
3,2A59BBDF5CDBA725,docked_bike,2020-04-07 12:50:19,2020-04-07 13:02:31,California Ave & Division St,216,Wood St & Augusta Blvd,657.0,41.9030,-87.6975,41.8992,-87.6722,member,12,12
4,27AD306C119C6158,docked_bike,2020-04-18 10:22:59,2020-04-18 11:15:54,Rush St & Hubbard St,125,Sheridan Rd & Lawrence Ave,323.0,41.8902,-87.6262,41.9695,-87.6547,casual,52,52


In [ ]:

df_all.drop(columns=['duration'], inplace=True)
df_all.info()

In [41]:
df_all.head(1)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration [min],is_weekend
0,A847FADBBC638E45,docked_bike,2020-04-26 17:45:14,2020-04-26 18:12:03,Eckhart Park,86,Lincoln Ave & Diversey Pkwy,152.0,41.8964,-87.661,41.9322,-87.6586,member,26,True


In [ ]:

df_all.groupby('member_casual')['duration [min]'].agg(['min', 'max',])

,min,max
member_casual,,
casual,-28995,98489
member,-29049,58720


In [49]:
# minimum has negative values, which is not possible, so we will remove those rows

negatives = df_all['duration [min]'] < 0
df_all = df_all[~negatives]
df_all.groupby('member_casual')['duration [min]'].agg(['min', 'max',])

,min,max
member_casual,,
casual,0,98489
member,0,58720


In [51]:
df_all.shape

(32165663, 15)

In [52]:
# id standardization
df_all['start_station_id'] = df_all['start_station_id'].astype(str)
df_all['end_station_id'] = df_all['end_station_id'].astype(str)

## Feature Engineering

To support behavioral analysis, additional time‑based features were derived while retaining the original timestamp columns:

- Trip start hour  
- Day of week  
- Weekend indicator  
- Month of trip  

The original datetime fields were preserved to maintain full temporal granularity for downstream analysis.

In [ ]:
df_all['start_hour'] = df_all['started_at'].dt.hour
df_all['day_of_week'] = df_all['started_at'].dt.day_name()
df_all['month'] = df_all['started_at'].dt.month_name()
# add a column for the day of the week
df_all['is_weekend'] = df_all['started_at'].dt.weekday >= 5
df_all.head(1)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration [min],is_weekend,start_hour,day_of_week,month
0,A847FADBBC638E45,docked_bike,2020-04-26 17:45:14,2020-04-26 18:12:03,Eckhart Park,86,Lincoln Ave & Diversey Pkwy,152.0,41.8964,-87.661,41.9322,-87.6586,member,26,True,17,Sunday,April


In [54]:
df_all.info()

<class 'pandas.DataFrame'>
Index: 32165663 entries, 0 to 32167288
Data columns (total 18 columns):
 #   Column              Dtype         
---  ------              -----         
 0   ride_id             str           
 1   rideable_type       str           
 2   started_at          datetime64[us]
 3   ended_at            datetime64[us]
 4   start_station_name  str           
 5   start_station_id    str           
 6   end_station_name    str           
 7   end_station_id      str           
 8   start_lat           float64       
 9   start_lng           float64       
 10  end_lat             float64       
 11  end_lng             float64       
 12  member_casual       str           
 13  duration [min]      int64         
 14  is_weekend          bool          
 15  start_hour          int32         
 16  day_of_week         str           
 17  month               str           
dtypes: bool(1), datetime64[us](2), float64(4), int32(1), int64(1), str(9)
memory usage: 4.2 GB


## Data Quality Validation

Data quality checks were performed to ensure the dataset was suitable for analysis:

- Missing values were assessed across all columns  
- Time‑based features were validated for realistic ranges  
- Geographic coordinates were checked to confirm valid latitude and longitude values  

Records with missing geographic coordinates were retained, as they do not impact time‑based behavioral analysis.

In [ ]:
# check for null values in the new columns
df_all[['duration [min]', 'is_weekend', 'start_hour', 'day_of_week', 'month']].isnull().sum()

,duration [min],is_weekend,start_hour,day_of_week,month
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
...,...,...,...,...,...
32167284,False,False,False,False,False
32167285,False,False,False,False,False
32167286,False,False,False,False,False
32167287,False,False,False,False,False


In [ ]:
# check for outliers in duration and start_hour
df_all[['duration [min]', 'start_hour']].describe()

,duration [min],start_hour
count,3.216566e+07,3.216566e+07
mean,1.895662e+01,1.413480e+01
std,1.571551e+02,4.937998e+00
min,0.000000e+00,0.000000e+00
25%,5.000000e+00,1.100000e+01
50%,1.000000e+01,1.500000e+01
75%,1.800000e+01,1.800000e+01
max,9.848900e+04,2.300000e+01


In [59]:
# check for null values in the entire dataframe
df_all.isna().sum().sort_values(ascending=False)

end_station_id        5145599
end_station_name      5144997
start_station_id      4867812
start_station_name    4867057
end_lng                 35242
end_lat                 35242
rideable_type               0
ride_id                     0
started_at                  0
ended_at                    0
start_lng                   0
start_lat                   0
member_casual               0
duration [min]              0
is_weekend                  0
start_hour                  0
day_of_week                 0
month                       0
dtype: int64

In [60]:
# check for invalid coordinates

invalid_coords = df_all[
    (~df_all['start_lat'].between(-90, 90)) |
    (~df_all['start_lng'].between(-180, 180)) |
    (~df_all['end_lat'].between(-90, 90)) |
    (~df_all['end_lng'].between(-180, 180))
]

invalid_coords.shape

(35242, 18)

In [ ]:
df_all[['start_lat', 'start_lng', 'end_lat', 'end_lng']].isna().sum()
# non values resembles with invalid coordinates, so we will not remove them, but we will keep in mind that they are not valid coordinates and should not be used for any analysis that requires valid coordinates.

start_lat        0
start_lng        0
end_lat      35242
end_lng      35242
dtype: int64

In [ ]:
# check for invalid coordinates, but only for rows where all coordinates are not null, because if there are null values, we cannot determine if the coordinates are valid or not, so we will keep those rows, but we will keep in mind that they are not valid coordinates and should not be used for any analysis that requires valid coordinates.
invalid_coords = df_all[
    df_all[['start_lat', 'start_lng', 'end_lat', 'end_lng']].notna().all(axis=1)
    &
    (
        (~df_all['start_lat'].between(-90, 90)) |
        (~df_all['start_lng'].between(-180, 180)) |
        (~df_all['end_lat'].between(-90, 90)) |
        (~df_all['end_lng'].between(-180, 180))
    )
]

invalid_coords.shape


(0, 18)

In [65]:
df_all[['start_lat', 'start_lng', 'end_lat', 'end_lng']].describe()
# chicago sanity check 
df_all[(df_all['start_lat'] >= 41.8) & (df_all['start_lat'] <= 42.1) & (df_all['start_lng'] >= -87.9) & (df_all['start_lng'] <= -87.5)].shape

(30787134, 18)

## Data Storage

After completing cleaning and validation, the processed dataset was loaded into Microsoft SQL Server using a Python‑based ETL pipeline.  
Data was inserted in chunks to efficiently handle the large dataset and support scalable analytical queries.



In [ ]:
# install required packages
pip install sqlalchemy pyodbc

In [ ]:
from sqlalchemy import create_engine

server = "Knok\SQLEXPRESS"          
database = "Analysis"

connection_string = (
    "mssql+pyodbc://@"
    f"{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string, fast_executemany=True)

In [70]:



# import dat do SQL Serveru
df_all.to_sql(
    name="cyclistic",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=100_000
)


-322

## Process Summary

- Raw data successfully consolidated and cleaned  
- Invalid records removed  
- Time‑based features derived  
- Data quality validated  
- Final dataset stored in SQL Server  

The dataset is now ready for analysis.